##  SQL Analysis

### Project Overview 
This notebook contains SQL-based analysis of the TMDB Movie dataset.

### Database 
SQLite

### Python Library 
SQLAlchemy

In [1]:
import pandas as pd

In [2]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///data/TMDB_database.db")

In [3]:
connection = engine.connect()

In [4]:
from sqlalchemy import inspect
inspector = inspect(engine)

tables = inspector.get_table_names()

print("Available Tables:", tables)
print("Total Tables:", len(tables))

Available Tables: ['cast', 'crew', 'genres', 'movie_genres', 'movie_keywords', 'movies']
Total Tables: 6


In [5]:
inspector = inspect(engine)

table_names = inspector.get_table_names()

print("Available Tables:")
for table in table_names:
    print(table)

print("Total Tables:", len(table_names)) 

Available Tables:
cast
crew
genres
movie_genres
movie_keywords
movies
Total Tables: 6


### Q1 .List all movies in the Action genre released after 2015, showing title and release date.




In [6]:
q1 = """
SELECT m.title, m.release_date
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g ON mg.genre_id = g.genre_id
WHERE g.genre_name = 'Action'
  AND m.release_date > '2015-12-31'
ORDER BY m.release_date
"""

q1_result = pd.read_sql(q1, engine)
q1_result

,title,release_date
0,The 5th Wave,2016-01-14
1,Kung Fu Panda 3,2016-01-23
2,Zoolander 2,2016-02-06
3,Deadpool,2016-02-09
4,Triple 9,2016-02-19
...,...,...
263,F1,2025-06-25
264,Superman,2025-07-09
265,The Fantastic 4: First Steps,2025-07-23
266,Predator: Badlands,2025-11-05


**Insights** 
Many Action movies were released after 2015, from The 5th Wave (2016) to F1 (2025). Action is still a popular genre.

### Q2 .Find the top 10 highest-grossing movies along with the genre(s) each belongs to. 

**Concept:** JOIN + ORDER BY + LIMIT



In [7]:
q2 = """
SELECT m.title, m.revenue, GROUP_CONCAT(g.genre_name, ', ') AS genres
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g ON mg.genre_id = g.genre_id
GROUP BY m.movie_id, m.title, m.revenue
ORDER BY m.revenue DESC
LIMIT 10
"""

q2_result = pd.read_sql(q2, engine)
q2_result

,title,revenue,genres
0,Avatar,2923706026,"Adventure, Action, Science Fiction"
1,Avengers: Endgame,2799439100,"Adventure, Action, Science Fiction"
2,Avatar: The Way of Water,2334484620,"Adventure, Action, Science Fiction"
3,Titanic,2264162353,"Drama, Romance"
4,Star Wars: The Force Awakens,2068223624,"Adventure, Action, Science Fiction"
5,Avengers: Infinity War,2052415039,"Adventure, Action, Science Fiction"
6,Spider-Man: No Way Home,1921206586,"Adventure, Action, Science Fiction"
7,Zootopia 2,1868208796,"Adventure, Animation, Comedy, Mystery, Family"
8,Inside Out 2,1698863816,"Adventure, Animation, Comedy, Family"
9,Jurassic World,1671537444,"Adventure, Thriller, Science Fiction"


**Insights** 
Avatar earned the most ($2.92B). Most top 10 movies are Adventure, Action or Science Fiction.

### Q3.Calculate the average budget and average revenue for each genre, sorted by average revenue descending.

**Concept:** JOIN + GROUP BY + AVG()



In [8]:
q3 = """
SELECT g.genre_name,
       ROUND(AVG(m.budget), 0) AS avg_budget,
       ROUND(AVG(m.revenue), 0) AS avg_revenue
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g ON mg.genre_id = g.genre_id
WHERE m.budget > 0 AND m.revenue > 0
GROUP BY g.genre_name
ORDER BY avg_revenue DESC
"""

q3_result = pd.read_sql(q3, engine)
q3_result

,genre_name,avg_budget,avg_revenue
0,Adventure,111393307.0,401451192.0
1,Animation,88282289.0,382103463.0
2,Family,88408767.0,357870775.0
3,Science Fiction,97864404.0,335689706.0
4,Action,94651416.0,309125778.0
5,Fantasy,89183665.0,307912995.0
6,Comedy,54652525.0,223039972.0
7,Music,42817021.0,205843825.0
8,Romance,37226946.0,177226878.0
9,Thriller,50844282.0,170049287.0


**Insights** 
Adventure has the highest average revenue ($401M). Drama earns much less ($147M) but also costs less to make.

### Q4. List the top 10 actors who have appeared in the most movies in the dataset.
    
**Concept:** (cast)GROUP BY + COUNT()



In [9]:
q4 = """
SELECT actor_name, COUNT(DISTINCT movie_id) AS movie_count
FROM cast
GROUP BY person_id, actor_name
ORDER BY movie_count DESC
LIMIT 10
"""

q4_result = pd.read_sql(q4, engine)
q4_result

,actor_name,movie_count
0,Samuel L. Jackson,39
1,Brad Pitt,38
2,Robert De Niro,38
3,Johnny Depp,37
4,Tom Hanks,33
5,Scarlett Johansson,32
6,Willem Dafoe,32
7,Mark Wahlberg,32
8,Morgan Freeman,31
9,Tom Cruise,31


**Insights** 
Samuel L. Jackson appeared in the most movies (39). He works in many genres, so he is cast often.

### Q5.Find all directors who have directed more than 3 movies, along with their movie count.

**Concept:** GROUP BY + HAVING

In [10]:
q5 = """
SELECT person_name AS director, COUNT(DISTINCT movie_id) AS movie_count
FROM crew
WHERE job = 'Director'
GROUP BY person_id, person_name
HAVING COUNT(DISTINCT movie_id) > 3
ORDER BY movie_count DESC
"""

q5_result = pd.read_sql(q5, engine)
q5_result

,director,movie_count
0,Steven Spielberg,27
1,Tim Burton,19
2,Ridley Scott,19
3,Martin Scorsese,16
4,Robert Zemeckis,15
...,...,...
212,Robert Eggers,4
213,Ryan Coogler,4
214,Andy Muschietti,4
215,Wes Ball,4


**Insights**
217 directors made more than 3 movies. Steven Spielberg has the most (27).

### Q6.List the top 10 most frequently used keywords across all movies.
    
**Concept:** GROUP BY + COUNT()



In [11]:
q6 = """
SELECT keyword_name, COUNT(*) AS usage_count
FROM movie_keywords
GROUP BY keyword_id, keyword_name
ORDER BY usage_count DESC
LIMIT 10
"""

q6_result = pd.read_sql(q6, engine)
q6_result

,keyword_name,usage_count
0,based on novel or book,449
1,sequel,416
2,duringcreditsstinger,296
3,aftercreditsstinger,242
4,murder,176
5,based on true story,166
6,new york city,161
7,villain,160
8,3d animation,155
9,based on comic,154


**Insights** 
“Based on novel or book” is the most common keyword (449), then “sequel” (416). Studios like to use known stories and franchises.

### Q7.Find movies where the budget was above the overall average budget but the revenue was below the overall average revenue.

**Concept:** Subquery comparison against AVG()



In [12]:
q7 = """
SELECT title, budget, revenue
FROM movies
WHERE budget > (SELECT AVG(budget) FROM movies WHERE budget > 0)
  AND revenue < (SELECT AVG(revenue) FROM movies WHERE revenue > 0)
  AND revenue > 0
ORDER BY budget DESC
"""

q7_result = pd.read_sql(q7, engine)
q7_result

,title,budget,revenue
0,The Marvels,274800000,206136825
1,Red One,250000000,185700759
2,Wake Up Dead Man: A Knives Out Mystery,210000000,4000000
3,Wonder Woman 1984,200000000,169601036
4,Luca,200000000,51074773
...,...,...,...
268,Rambo III,63000000,189015611
269,The Equalizer 2,62000000,190400157
270,Æon Flux,62000000,53321673
271,Shooter,61000000,95700000


**Insights** 
Many big-budget movies earned less than average, like The Marvels ($275M budget, $206M revenue). A high budget does not guarantee high revenue.

### Q8.List all actors who have appeared in a movie directed by a specific director (e.g., Christopher Nolan).

**Concept:** JOIN (cast ⋈ crew )



In [13]:
q8 = """
SELECT DISTINCT c.actor_name
FROM cast c
JOIN crew d ON c.movie_id = d.movie_id
WHERE d.job = 'Director'
  AND d.person_name = 'Christopher Nolan'
ORDER BY c.actor_name
"""

q8_result = pd.read_sql(q8, engine)
q8_result

,actor_name
0,Aaron Eckhart
1,Al Pacino
2,Alon Aboutboul
3,Andy Serkis
4,Aneurin Barnard
...,...
79,Tom Hardy
80,Tom Wilkinson
81,Topher Grace
82,Wes Bentley


**Insights** 
84 actors have worked with Christopher Nolan, like Al Pacino and Tom Hardy. He works with many different actors.

### Q9.Find the genre with the highest average rating, considering only genres with at least 20 movies.

**Concept:** JOIN + GROUP BY + HAVING + AVG()



In [14]:
q9 = """
SELECT g.genre_name,
       ROUND(AVG(m.vote_average), 2) AS avg_rating,
       COUNT(*) AS movie_count
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g ON mg.genre_id = g.genre_id
GROUP BY g.genre_name
HAVING COUNT(*) >= 20
ORDER BY avg_rating DESC
LIMIT 1
"""

q9_result = pd.read_sql(q9, engine)
q9_result

,genre_name,avg_rating,movie_count
0,War,7.36,85


**Insights** 
War has the highest average rating (7.36) among genres with 20+ movies.

### Q10.List the top 10 movies with the highest number of keywords tagged to them. 

**Concept:** JOIN + GROUP BY



In [15]:
q10 = """
SELECT m.title, COUNT(k.keyword_id) AS keyword_count
FROM movies m
JOIN movie_keywords k ON m.movie_id = k.movie_id
GROUP BY m.movie_id, m.title
ORDER BY keyword_count DESC
LIMIT 10
"""

q10_result = pd.read_sql(q10, engine)
q10_result

,title,keyword_count
0,Smile 2,98
1,Taken 3,70
2,Silent Hill,63
3,Twelve Monkeys,57
4,War for the Planet of the Apes,54
5,The Shining,52
6,Dawn of the Planet of the Apes,51
7,How to Train Your Dragon,50
8,How to Train Your Dragon: The Hidden World,49
9,Pacific Rim: Uprising,49


**Insights** 
Smile 2 has the most keywords (98). More keywords only means more tags, not a better movie.